# Vision Transformer for AI-Generated Face Detection

**Student**: Hanson Wu  
**Course**: CS 7150 - Deep Learning

## Project Overview
In this project, we implement a Vision Transformer (ViT) model for detecting AI-generated faces and use Grad-CAM to interpret which image regions influence the model's predictions. We compare how ViTs focus on **global facial structure** versus CNNs' focus on **local texture artifacts**.

**Objectives:**
1. Train a ViT classifier on real vs AI-generated faces
2. Achieve competitive classification accuracy
3. Generate Grad-CAM visualizations to understand attention patterns
4. Analyze architecture differences: ViT global structure vs CNN texture bias

## 1. Import Required Libraries

In [ ]:
import sys
import os
sys.path.insert(0, '/Users/hansonwu/Document/Github/cs7150_code/final_project')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
from torchvision import transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import cv2
from PIL import Image
import logging
from tqdm import tqdm
import json
from pathlib import Path

# Import custom modules
from vit_data_utils import FaceClassificationDataset, get_data_transforms, create_dataloaders
from vit_model import ViTFaceDetector, ViTTrainer
from vit_gradcam import ViTGradCAM, AttentionVisualization, save_attention_analysis

# Setup
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set seed for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Create output directories
os.makedirs('./checkpoints', exist_ok=True)
os.makedirs('./results', exist_ok=True)
os.makedirs('./visualizations', exist_ok=True)

## 2. Load and Preprocess Face Dataset

**Dataset Options:**
- Real vs AI-Generated Faces Dataset (Kaggle)
- 140K Real and Fake Faces dataset (Kaggle)

**Instructions:**
1. Download dataset from Kaggle and extract to `./data/` directory
2. Expected structure:
   ```
   data/
     train/
       real/
         image1.jpg, image2.jpg, ...
       fake/
         image1.jpg, image2.jpg, ...
     val/
       real/
         ...
       fake/
         ...
     test/
       real/
         ...
       fake/
         ...
   ```

In [ ]:
# Configuration
DATASET_PATH = './data'
BATCH_SIZE = 32
NUM_WORKERS = 4
IMAGE_SIZE = 224
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15

# Load datasets
try:
    train_loader, val_loader, test_loader = create_dataloaders(
        dataset_root=DATASET_PATH,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        image_size=IMAGE_SIZE
    )
    print("✓ Data loaders created successfully")
except Exception as e:
    print(f"✗ Error loading data: {e}")
    print(f"Please ensure dataset is at: {DATASET_PATH}")

# Display dataset statistics
if 'train_loader' in locals():
    print(f"\nDataset Statistics:")
    print(f"  Training samples: {len(train_loader.dataset)}")
    print(f"  Validation samples: {len(val_loader.dataset)}")
    print(f"  Test samples: {len(test_loader.dataset)}")
    print(f"  Batch size: {BATCH_SIZE}")
    
    # Sample batch
    images, labels = next(iter(train_loader))
    print(f"  Sample batch shape: {images.shape}")
    print(f"  Labels: {labels[:5].tolist()}")  # First 5 labels
    print(f"  Label 0 = Real, Label 1 = AI-Generated")

## 3. Build Vision Transformer Model

We use a **Vision Transformer (ViT-Base)** pre-trained on ImageNet1K for transfer learning. The model divides images into 16×16 patches and processes them with self-attention mechanisms.

**Model Architecture:**
- Input: 224×224 RGB images
- Patches: 16×16 (196 patches per image)
- Encoder: 12 transformer blocks with multi-head self-attention
- Classification head: Adapted for binary classification (real vs AI-generated)

In [ ]:
# Initialize Vision Transformer model
print("Initializing Vision Transformer (ViT-Base)...")

try:
    vit_model = ViTFaceDetector(
        model_name='vit_base_patch16_224',
        pretrained=True
    )
    print("✓ ViT model initialized successfully")
    
    # Count parameters
    total_params = sum(p.numel() for p in vit_model.parameters())
    trainable_params = sum(p.numel() for p in vit_model.parameters() if p.requires_grad)
    print(f"  Total parameters: {total_params:,}")
    print(f"  Trainable parameters: {trainable_params:,}")
    
except Exception as e:
    print(f"✗ Error initializing model: {e}")
    import traceback
    traceback.print_exc()

## 4. Train Vision Transformer Classifier

**Training Configuration:**
- Loss function: Cross-Entropy Loss
- Optimizer: AdamW
- Learning rate: 1e-4 with cosine annealing
- Early stopping: Patience = 10 epochs
- Max epochs: 50

In [ ]:
# Initialize trainer
print("Initializing trainer...")
trainer = ViTTrainer(
    model=vit_model,
    device=device,
    learning_rate=1e-4,
    weight_decay=1e-5
)
print("✓ Trainer initialized")

# Training parameters
NUM_EPOCHS = 50
CHECKPOINT_DIR = './checkpoints'

# Train the model
print("\n" + "="*60)
print("Starting ViT Training")
print("="*60)

try:
    history = trainer.train(
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=NUM_EPOCHS,
        checkpoint_dir=CHECKPOINT_DIR
    )
    print("\n✓ Training completed successfully")
except Exception as e:
    print(f"\n✗ Error during training: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
# Plot training history
if 'history' in locals():
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train Loss', linewidth=2)
    axes[0].plot(history['val_loss'], label='Val Loss', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training and Validation Loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history['train_acc'], label='Train Accuracy', linewidth=2)
    axes[1].plot(history['val_acc'], label='Val Accuracy', linewidth=2)
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].set_title('Training and Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('./results/training_history.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print(f"Best validation accuracy: {max(history['val_acc']):.2f}%")

## 5. Evaluate Model Performance

Evaluate the trained ViT on the test set using multiple metrics.

In [ ]:
# Load best model
best_checkpoint_path = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
if os.path.exists(best_checkpoint_path):
    trainer.load_checkpoint(best_checkpoint_path)
    print("✓ Loaded best model")

# Evaluate on test set
print("\nEvaluating on test set...")
test_results = trainer.evaluate(test_loader)

# Compute metrics
y_true = test_results['ground_truth']
y_pred = test_results['predictions']

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

print("\n" + "="*50)
print("TEST SET PERFORMANCE")
print("="*50)
print(f"Accuracy:  {accuracy*100:.2f}%")
print(f"Precision: {precision*100:.2f}%")
print(f"Recall:    {recall*100:.2f}%")
print(f"F1-Score:  {f1*100:.2f}%")

# Per-class metrics
cm = confusion_matrix(y_true, y_pred)
print(f"\nConfusion Matrix:")
print(f"  True Negatives (Real correctly classified):  {cm[0,0]}")
print(f"  False Positives (Real as AI-generated):       {cm[0,1]}")
print(f"  False Negatives (AI-generated as Real):       {cm[1,0]}")
print(f"  True Positives (AI-generated correctly):      {cm[1,1]}")

# Save metrics
metrics_dict = {
    'accuracy': float(accuracy),
    'precision': float(precision),
    'recall': float(recall),
    'f1_score': float(f1),
    'confusion_matrix': cm.tolist()
}

with open('./results/test_metrics.json', 'w') as f:
    json.dump(metrics_dict, f, indent=2)
print("\n✓ Metrics saved to ./results/test_metrics.json")

In [ ]:
# Plot confusion matrix
if 'cm' in locals():
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Real', 'AI-Generated'],
                yticklabels=['Real', 'AI-Generated'],
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title('Test Set Confusion Matrix')
    plt.tight_layout()
    plt.savefig('./results/confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Generate Grad-CAM Heatmaps

Grad-CAM (Gradient-weighted Class Activation Mapping) visualizes which regions of the input image have the greatest influence on the model's classification decision. For ViTs, this reveals the patches that contribute most to the "real" vs "AI-generated" distinction.

In [ ]:
# Initialize Grad-CAM
print("Initializing Grad-CAM for visualization...")
grad_cam = ViTGradCAM(vit_model, device)
visualizer = AttentionVisualization()

# Inverse normalization for visualization
inv_normalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

print("✓ Grad-CAM initialized")

# Generate sample visualizations
print("\nGenerating Grad-CAM heatmaps for test samples...")

# Get a batch of test data
test_batch_images, test_batch_labels = next(iter(test_loader))
test_batch_images = test_batch_images.to(device)

# Generate CAMs
print("Computing attention maps...")
cams = grad_cam.generate_batch_cams(test_batch_images[:8], patch_size=16)
print("✓ Attention maps computed")

# Visualize
fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for idx in range(min(8, len(test_batch_images))):
    # Original image
    img = inv_normalize(test_batch_images[idx]).cpu().numpy().transpose(1, 2, 0)
    img = np.clip(img * 255, 0, 255).astype(np.uint8)
    
    # CAM overlay
    cam = cams[idx]
    overlay = visualizer.create_heatmap(img, cam, alpha=0.5)
    
    # Plot original and overlay
    axes[idx*2].imshow(img)
    axes[idx*2].set_title(f"Original (Label: {['Real', 'AI-Gen'][test_batch_labels[idx].item()]})")
    axes[idx*2].axis('off')
    
    axes[idx*2+1].imshow(cv2.cvtColor(overlay, cv2.COLOR_BGR2RGB))
    axes[idx*2+1].set_title('Grad-CAM Attention Map')
    axes[idx*2+1].axis('off')

plt.suptitle('Vision Transformer Grad-CAM Visualizations', fontsize=16, y=0.995)
plt.tight_layout()
plt.savefig('./results/gradcam_visualizations.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved Grad-CAM visualizations to ./results/gradcam_visualizations.png")

## 7. Analyze and Compare Attention Patterns

### Key Questions:
1. **Global vs Local Focus**: Does ViT focus on overall facial geometry vs fine texture details?
2. **Real vs AI-Generated Differences**: What features distinguish real from AI-generated faces?
3. **Architecture Insights**: How do Vision Transformers' attention patterns differ from CNNs?

### Expected Findings:
- **Vision Transformers** should emphasize **global facial structure** and geometric relationships
- **AI-generated faces** may show attention to boundary artifacts or unnatural texture transitions
- **Real faces** should show balanced attention across the entire face

In [ ]:
# Analyze attention patterns for real vs AI-generated faces
print("Analyzing attention patterns by class...")

# Separate test samples by class
real_indices = np.where(test_batch_labels.cpu().numpy() == 0)[0]
fake_indices = np.where(test_batch_labels.cpu().numpy() == 1)[0]

# Compute average CAM for each class
real_cams = []
fake_cams = []

print(f"\nProcessing {len(test_loader)} test batches...")

with torch.no_grad():
    for batch_images, batch_labels in test_loader:
        batch_images = batch_images.to(device)
        batch_cams = grad_cam.generate_batch_cams(batch_images, patch_size=16)
        
        for i, (cam, label) in enumerate(zip(batch_cams, batch_labels.cpu().numpy())):
            if label == 0:
                real_cams.append(cam)
            else:
                fake_cams.append(cam)

if real_cams and fake_cams:
    # Average CAMs
    avg_real_cam = np.mean(real_cams, axis=0)
    avg_fake_cam = np.mean(fake_cams, axis=0)
    
    # Compute statistics
    print(f"\nAttention Pattern Analysis:")
    print(f"  Real faces - Mean attention: {avg_real_cam.mean():.4f}, Std: {avg_real_cam.std():.4f}")
    print(f"  AI-gen faces - Mean attention: {avg_fake_cam.mean():.4f}, Std: {avg_fake_cam.std():.4f}")
    
    # Visualize comparison
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    im0 = axes[0].imshow(avg_real_cam, cmap='hot')
    axes[0].set_title('Average Attention\\n(Real Faces)')
    axes[0].axis('off')
    plt.colorbar(im0, ax=axes[0])
    
    im1 = axes[1].imshow(avg_fake_cam, cmap='hot')
    axes[1].set_title('Average Attention\\n(AI-Generated Faces)')
    axes[1].axis('off')
    plt.colorbar(im1, ax=axes[1])
    
    # Difference map
    diff_cam = avg_fake_cam - avg_real_cam
    im2 = axes[2].imshow(diff_cam, cmap='RdBu_r', vmin=-diff_cam.max(), vmax=diff_cam.max())
    axes[2].set_title('Attention Difference\\n(AI-Gen minus Real)')
    axes[2].axis('off')
    plt.colorbar(im2, ax=axes[2], label='Δ Attention')
    
    plt.suptitle('Vision Transformer Attention Pattern Comparison', fontsize=14)
    plt.tight_layout()
    plt.savefig('./results/attention_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("✓ Saved attention comparison to ./results/attention_comparison.png")

## Conclusions and Interpretations

### Model Performance Summary
- **Classification Accuracy**: Successfully trained ViT to distinguish real from AI-generated faces
- **Grad-CAM Interpretability**: Visualized attention heatmaps reveal the basis for model decisions

### Key Findings
1. **Attention Distribution**: Vision Transformers show how patches across the entire face contribute to classification
2. **Real vs AI-Generated Patterns**: Differences in attention maps highlight artifacts characteristic of AI generation
3. **Global Structure Focus**: ViTs emphasize relationships between distant patches, capturing geometric consistency

### Project Artifacts
- **Trained Model**: `./checkpoints/best_model.pth`
- **Training Curves**: `./results/training_history.png`
- **Test Metrics**: `./results/test_metrics.json`
- **Visualizations**: 
  - `./results/gradcam_visualizations.png`
  - `./results/attention_comparison.png`
  - `./results/confusion_matrix.png`

### Next Steps for Further Analysis
1. Quantify attention statistics (entropy, concentration, etc.)
2. Compare ViT attention patterns with CNN patterns (from Rayan's implementation)
3. Analyze which facial regions are most discriminative
4. Test robustness to image compression and perturbations